In [3]:

import pandas as pd
import numpy as np
import datetime
import os
from io import StringIO


# =====================================================
# STEP 1: LOGGER CLASS
# =====================================================
class AA_Logger:
    def __init__(self, vLogfilePath, vSystemMachine, vProcessName):
        self.logFilePath = vLogfilePath
        self.scriptName = "Data_Cleaning.py"
        self.systemMachine = vSystemMachine
        self.processName = vProcessName
    def log(self, statusCode, message):
        status_map = {0: "Info",1: "Error",2: "Warn",3: "Debug"}
        status = status_map.get(statusCode, "Info")
        now = datetime.datetime.now().strftime("%m/%d/%Y %H:%M:%S")
        line = (f"{statusCode}, {status}, {now}, "f"{self.systemMachine}, {self.processName}, "f"{self.scriptName}, {message}")
        with open(self.logFilePath, "a", encoding="utf-8") as f:
            f.write(line + "\n")
        print(line)

# =====================================================
# STEP 2: DATE PARSING FUNCTION
# =====================================================
def parse_order_date(value):
    if pd.isna(value):
        return pd.NaT
    if isinstance(value, (datetime.datetime, datetime.date)):
        return pd.to_datetime(value)
    value = str(value).strip()
    formats = [
        "%Y-%m-%d",
        "%d/%m/%Y",
        "%m-%d-%Y",
        "%d-%b-%Y",
        "%Y/%m/%d"
    ]
    for fmt in formats:
        try:
            return pd.Timestamp(datetime.datetime.strptime(value, fmt))
        except ValueError:
            continue
    return pd.NaT #Not a Time

# =====================================================
# STEP 3: MAIN FUNCTION
# =====================================================
def main(vList):

    vLogfilePath = vList[0]
    vSystemMachine = vList[1]
    vProcessName = vList[2]
    vProjectFolder = vList[3]
    vSheetName = vList[4]

    logger = AA_Logger(vLogfilePath,vSystemMachine,vProcessName)
    try:
        logger.log(0, "Data Cleaning Execution Started")
        # -------------------------------------------------
        # STEP 3.1: LOAD RAW DATASET
        # -------------------------------------------------
        vExcelFile = os.path.join(vProjectFolder,"Data", "Raw", "ecommerce_raw.xlsx")
        df = pd.read_excel(vExcelFile,sheet_name=vSheetName)
        # Check required columns
        required_columns = [
            "Order ID",
            "Customer ID",
            "Product Name",
            "Category",
            "Quantity",
            "Unit Price",
            "Total Amount",
            "Order Date",
            "Payment Mode",
            "Delivery Status",
            "City"
        ]
        missing_columns = [
            col for col in required_columns
            if col not in df.columns
        ]
        if missing_columns:
            raise ValueError(f"Missing required columns: {missing_columns}")
            
        original_columns = df.columns.tolist()
        # Preserve the original data for duplicate analysis
        original_df = df[original_columns].copy()
        # Detect exact duplicates in the original data
        duplicate_mask = original_df.duplicated(subset=original_columns,keep="first")

        logger.log(0, f"Dataset Shape: {df.shape}")
        logger.log(0, f"First Five Records:\n{df.head()}")

        # Dataset information
        buffer = StringIO()
        df.info(buf=buffer)

        logger.log(0, buffer.getvalue())

        # -------------------------------------------------
        # STEP 3.2: MISSING VALUE ANALYSIS BEFORE CLEANING
        # -------------------------------------------------
        missing_count = df.isnull().sum()
        missing_percentage = (df.isnull().sum() / len(df)) * 100

        missing_report = pd.DataFrame({
            "Missing Values": missing_count,
            "Missing Percentage": missing_percentage.round(2)
        })

        logger.log(0,f"Missing Value Report Before Cleaning:\n{missing_report}")

        # Record missing columns for each original row
        df["Missing Fields"] = df.isnull().apply(lambda row: ", ".join(row.index[row]),axis=1)

        # -------------------------------------------------
        # STEP 3.3: PRESERVE ORIGINAL VALUES FOR VALIDATION
        # -------------------------------------------------
        original_quantity = df["Quantity"].copy()
        original_unit_price = df["Unit Price"].copy()
        original_total_amount = df["Total Amount"].copy()
        original_order_date = df["Order Date"].copy()

        # -------------------------------------------------
        # STEP 3.4: NUMERIC DATA CONVERSION
        # -------------------------------------------------
        numeric_columns = ["Quantity","Unit Price","Total Amount"]
        invalid_numeric = {}
        for col in numeric_columns:
            converted_values = pd.to_numeric(df[col],errors="coerce")
            # Identify non-empty values that could not be converted
            invalid_numeric[col] = (df[col].notna() & converted_values.isna())
            df[col] = converted_values

        # -------------------------------------------------
        # STEP 3.5: HANDLE MISSING CATEGORICAL VALUES
        # -------------------------------------------------
        categorical_cols = [
            "Customer ID",
            "Product Name",
            "Category",
            "Payment Mode",
            "Delivery Status",
            "City"
        ]
        for col in categorical_cols:
            df[col] = df[col].fillna("Unknown")

        # Standardize text spacing and capitalization
        standardize_columns = [
            "Category",
            "Payment Mode",
            "Delivery Status",
            "City"
        ]
        for col in standardize_columns:
            df[col] = (df[col].astype(str).str.strip().str.title())

        # Remove extra spaces from product names
        df["Product Name"] = (df["Product Name"].astype(str).str.strip())

        # -------------------------------------------------
        # STEP 3.6: RECALCULATE MISSING TOTAL AMOUNT
        # -------------------------------------------------
        valid_inputs = (
            df["Quantity"].notna()
            & df["Unit Price"].notna()
            & (df["Quantity"] > 0)
            & (df["Unit Price"] > 0)
        )
        missing_total = (df["Total Amount"].isna()& valid_inputs)

        df.loc[missing_total, "Total Amount"] = (
            df.loc[missing_total, "Quantity"]
            * df.loc[missing_total, "Unit Price"]
        )

        # -------------------------------------------------
        # STEP 3.7: CLEANING STATUS
        # -------------------------------------------------
        df["Cleaning Status"] = np.where(
            df["Missing Fields"] == "",
            "No Missing Values Initially",
            "Missing Values Handled or Flagged"
        )

        # -------------------------------------------------
        # STEP 3.8: MISSING VALUE REPORT AFTER CLEANING
        # -------------------------------------------------
        report = pd.DataFrame({
            "Column": df.columns,
            "Missing Values After Cleaning":
                df.isnull().sum().values,
            "Missing Percentage After Cleaning":
                (df.isnull().mean() * 100).round(2).values
        })

        logger.log(0,f"Missing Value Report After Cleaning:\n{report}")

        # =================================================
        # STEP 4: DATA VALIDATION
        # =================================================
        df["Validation Errors"] = ""
        def add_error(mask, message):
            mask = mask.fillna(False)
            df.loc[mask, "Validation Errors"] = (
                df.loc[mask, "Validation Errors"]
                .apply(lambda x: x + "; " if x else "")
                + message
            )

        # -------------------------------------------------
        # 4.1: INVALID NUMERIC VALUES
        # -------------------------------------------------
        for col in numeric_columns:
            add_error(invalid_numeric[col],f"Invalid {col}")

        # -------------------------------------------------
        # 4.2: INVALID QUANTITY
        # -------------------------------------------------
        add_error(
            df["Quantity"].notna()
            & (df["Quantity"] <= 0),
            "Invalid Quantity"
        )

        # -------------------------------------------------
        # 4.3: INVALID UNIT PRICE
        # -------------------------------------------------
        add_error(
            df["Unit Price"].notna()
            & (df["Unit Price"] <= 0),
            "Invalid Unit Price"
        )

        # -------------------------------------------------
        # 4.4: MISSING NUMERIC VALUES
        # -------------------------------------------------
        add_error(df["Quantity"].isna(),"Missing Quantity")
        add_error(df["Unit Price"].isna(),"Missing Unit Price")
        add_error(df["Total Amount"].isna(),"Missing Total Amount")

        # -------------------------------------------------
        # 4.5: TOTAL AMOUNT VALIDATION
        # -------------------------------------------------

        valid_amount_inputs = (
            df["Quantity"].notna()
            & df["Unit Price"].notna()
            & (df["Quantity"] > 0)
            & (df["Unit Price"] > 0)
        )

        amount_mismatch = (
            valid_amount_inputs
            & df["Total Amount"].notna()
            & ~np.isclose(
                df["Total Amount"],
                df["Quantity"] * df["Unit Price"],
                rtol=0,
                atol=0.01
            )
        )
        add_error(amount_mismatch,"Total Amount Mismatch")

        # -------------------------------------------------
        # 4.6: ORDER DATE VALIDATION
        # -------------------------------------------------
        parsed_dates = original_order_date.apply(parse_order_date)
        add_error(original_order_date.isna(),"Missing Order Date")
        add_error(
            original_order_date.notna()
            & parsed_dates.isna(),
            "Invalid Order Date"
        )
        df["Order Date"] = parsed_dates
        # -------------------------------------------------
        # 4.7: ORDER ID VALIDATION
        # -------------------------------------------------
        add_error(df["Order ID"].isna(),"Missing Order ID")

        # -------------------------------------------------
        # 4.8: EXACT DUPLICATE VALIDATION
        # -------------------------------------------------
        add_error(duplicate_mask,"Exact Duplicate Record")
        duplicate_records = df[duplicate_mask].copy()

        # -------------------------------------------------
        # 4.9: REPEATED ORDER ID ANALYSIS
        # -------------------------------------------------
        repeated_orders = df[
            df.duplicated(
                subset=["Order ID"],
                keep=False
            )
        ].copy()

        # =================================================
        # STEP 5: SEPARATE VALID AND INVALID RECORDS
        # =================================================
        invalid_rows = df[
            df["Validation Errors"] != ""
        ].copy()

        valid_rows = df[
            df["Validation Errors"] == ""
        ].copy()

        logger.log(0,f"Total Records: {len(df)}")
        logger.log(0,f"Valid Records: {len(valid_rows)}")
        logger.log(0,f"Invalid Records: {len(invalid_rows)}")
        logger.log(0,f"Exact Duplicate Records: {len(duplicate_records)}")
        logger.log(0,f"Repeated Order ID Records: {len(repeated_orders)}")

        validation_summary = (
            invalid_rows["Validation Errors"]
            .value_counts()
            .to_frame("Record Count")
        )
        logger.log(0,f"Validation Error Summary:\n{validation_summary}")

        # =================================================
        # STEP 6: EXPORT FINAL EXCEL WORKBOOK
        # =================================================
        output_folder = os.path.join(vProjectFolder,"Data","Cleaned")
        output_file = os.path.join(output_folder,"Ecommerce_Cleaning_Output.xlsx")

        with pd.ExcelWriter(output_file,engine="openpyxl") as writer:
            valid_rows.to_excel(writer,sheet_name="Cleaned_Data",index=False)
            invalid_rows.to_excel(writer,sheet_name="Invalid_Records",index=False)
            duplicate_records.to_excel(writer,sheet_name="Duplicate_Records",index=False)
            repeated_orders.to_excel(writer,sheet_name="Repeated_Order_IDs",index=False)
            missing_report.to_excel(writer,sheet_name="Missing_Before")

            report.to_excel(writer,sheet_name="Missing_After",index=False)
            validation_summary.to_excel(writer,sheet_name="Validation_Summary")
            df.to_excel(writer,sheet_name="Data_With_Validation",index=False)

        logger.log(0,f"Output Workbook Saved: {output_file}")
        logger.log(0,"Data Cleaning Execution Completed Successfully")
    except Exception as e:
        logger.log(1,f"Execution Failed: {str(e)}")
        raise


if __name__ == "__main__":
    vList = [
        r"C:\Sharmila\Data Analyst\Weintern\E-Commerce Sales Data Analyst\Logs\E-Commerce Dataset Cleaning.log",
        "0C8PI8MG",
        "E-Commerce Dataset Cleaning",
        r"C:\Sharmila\Data Analyst\Weintern\E-Commerce Sales Data Analyst",
        "Raw_Ecommerce_Data"
    ]

    main(vList)


0, Info, 11/07/2026 23:51:36, 0C8PI8MG, E-Commerce Dataset Cleaning, Data_Cleaning.py, Data Cleaning Execution Started
0, Info, 11/07/2026 23:51:46, 0C8PI8MG, E-Commerce Dataset Cleaning, Data_Cleaning.py, Dataset Shape: (50000, 11)
0, Info, 11/07/2026 23:51:46, 0C8PI8MG, E-Commerce Dataset Cleaning, Data_Cleaning.py, First Five Records:
    Order ID Customer ID          Product Name        Category  Quantity  \
0  ORD100000   CUST08518  Bluetooth Headphones     Electronics       1.0   
1  ORD100001   CUST05010              Yoga Mat          Sports       5.0   
2  ORD100002         NaN             Face Wash          Beauty       3.0   
3  ORD100003   CUST06901          Water Bottle  Home & Kitchen       5.0   
4  ORD100004   CUST01428          Water Bottle  Home & Kitchen       3.0   

   Unit Price  Total Amount  Order Date      Payment Mode Delivery Status  \
0     1888.95       1888.95  2024-05-24            Wallet      Processing   
1      878.90       4394.50  2025-12-23          